# SceneVerse data check

Exploratory pass over the freshly-downloaded SceneVerse files
(`dataset/download_sceneverse.sh` / `.sbatch`) before writing any real
preprocessing. Two files so far:

- `source_data/sceneverse/scannet_scene_cap.json` — ScanNet scene-level captions
- `source_data/sceneverse/3rscan.zip` — unknown internal structure, not extracted yet

Goal: get real counts (not estimates) for "how many usable scenes does this
add" so we can decide if the scene-level-caption pretraining data pool is
big enough, and cross-reference against what's already confirmed
(`leo_annotations` 3RScan `scene_caption`, MMScan `Caption_region`).

**Adjust the path config in the first cell to match your actual cluster layout before running.**

In [1]:
import json, zipfile
from pathlib import Path

# --- path config -- adjust these to your actual layout ---
ROOT = Path("/glob/g01-cache/pf/Yushuo/vjepa201")
SCENEVERSE_ROOT = ROOT / "source_data/sceneverse"
SCANNET_SCENE_CAP = SCENEVERSE_ROOT / "scannet_scene_cap.json"
THREERSCAN_ZIP = SCENEVERSE_ROOT / "3rscan.zip"

LEO_ANNO = ROOT / "source_data/leo_annotations/annotations"
SCANNET_POSED = ROOT / "source_data/scannet/posed_images"
THREERSCAN_ROOT = ROOT / "source_data/3rscan"

for name, p in [
    ("scannet_scene_cap", SCANNET_SCENE_CAP),
    ("3rscan.zip", THREERSCAN_ZIP),
    ("leo_anno", LEO_ANNO),
    ("scannet_posed", SCANNET_POSED),
    ("3rscan_raw", THREERSCAN_ROOT),
]:
    print(("OK  " if p.exists() else "MISSING  ") + f"{name:<18} {p}")

OK  scannet_scene_cap  /glob/g01-cache/pf/Yushuo/vjepa201/source_data/sceneverse/scannet_scene_cap.json
OK  3rscan.zip         /glob/g01-cache/pf/Yushuo/vjepa201/source_data/sceneverse/3rscan.zip
OK  leo_anno           /glob/g01-cache/pf/Yushuo/vjepa201/source_data/leo_annotations/annotations
OK  scannet_posed      /glob/g01-cache/pf/Yushuo/vjepa201/source_data/scannet/posed_images
OK  3rscan_raw         /glob/g01-cache/pf/Yushuo/vjepa201/source_data/3rscan


## 1. `3rscan.zip` -- list contents (not extracted)

We don't know what's inside yet: could be a straight repackage of leo's
`3rscan_scenecap_{train,val}.json`, could be something new (per-region,
different captions, extra metadata). List every entry + total size first,
extract nothing to disk.

In [2]:
with zipfile.ZipFile(THREERSCAN_ZIP) as z:
    infos = z.infolist()

total_bytes = sum(i.file_size for i in infos)
print(f"{len(infos)} entries, {total_bytes / 1e6:.1f} MB uncompressed total\n")

# group by extension to see what kinds of files are in here
from collections import Counter
ext_counts = Counter(Path(i.filename).suffix for i in infos)
print("by extension:", dict(ext_counts))

print("\nfirst 30 entries:")
for i in infos[:30]:
    print(f"  {i.file_size:>12} bytes  {i.filename}")

2785 entries, 4222.9 MB uncompressed total

by extension: {'': 6, '.pth': 2762, '.json': 13, '.txt': 4}

first 30 entries:
             0 bytes  3RScan/
             0 bytes  3RScan/scan_data/
             0 bytes  3RScan/scan_data/instance_id_to_label/
           815 bytes  3RScan/scan_data/instance_id_to_label/752cc581-920c-26f5-8e51-9520e8441118.pth
           687 bytes  3RScan/scan_data/instance_id_to_label/742e8f15-be0a-294e-9ebb-6c72dbcb9662.pth
           687 bytes  3RScan/scan_data/instance_id_to_label/c7895f29-339c-2d13-83e9-90dbe61fa8be.pth
           687 bytes  3RScan/scan_data/instance_id_to_label/1d2f8510-d757-207c-8c48-3684433860e1.pth
           751 bytes  3RScan/scan_data/instance_id_to_label/feefd90b-9d00-2ce5-8119-b936443cc39b.pth
           751 bytes  3RScan/scan_data/instance_id_to_label/7ab2a9bf-ebc6-2056-8bd5-903a96eb7e99.pth
           751 bytes  3RScan/scan_data/instance_id_to_label/c92fb5b1-f771-2064-8492-1233552bf94d.pth
           751 bytes  3RScan/scan_data/

## 2. Peek inside the zip's json/text files without extracting

For every `.json` (or other small text-like) entry, read it straight from
the zip (`ZipFile.open`, in-memory) and print its top-level structure --
dict keys / list length / one example item. Skips anything over ~20MB
(read as raw bytes for a byte-count instead) so we don't accidentally load
something huge into memory.

In [3]:
SIZE_CAP = 20_000_000  # 20MB

def describe(obj, max_items=6):
    if isinstance(obj, dict):
        keys = list(obj.keys())
        print(f"  dict, {len(keys)} keys, first keys: {keys[:max_items]}")
        if keys:
            k0 = keys[0]
            print(f"  obj[{k0!r}] = {str(obj[k0])[:400]}")
    elif isinstance(obj, list):
        print(f"  list, {len(obj)} items")
        if obj:
            print(f"  obj[0] = {str(obj[0])[:400]}")
    else:
        print(f"  {type(obj)}: {str(obj)[:400]}")

with zipfile.ZipFile(THREERSCAN_ZIP) as z:
    text_like = [i for i in z.infolist() if i.filename.lower().endswith((".json", ".txt", ".csv"))]
    print(f"{len(text_like)} json/txt/csv entries found\n")
    for i in text_like:
        print(f"=== {i.filename}  ({i.file_size} bytes) ===")
        if i.file_size > SIZE_CAP:
            print(f"  skipped (over {SIZE_CAP/1e6:.0f}MB cap)")
            continue
        raw = z.read(i.filename)
        if i.filename.lower().endswith(".json"):
            try:
                obj = json.loads(raw)
                describe(obj)
            except json.JSONDecodeError as e:
                print(f"  not valid single-document JSON ({e}); first 300 bytes:")
                print(" ", raw[:300])
        else:
            print(" ", raw[:300])
        print()

17 json/txt/csv entries found

=== 3RScan/annotations/ssg_obj_caption_gpt.json  (15330379 bytes) ===
  list, 38726 items
  obj[0] = {'item_id': '352e9c30-69fb-27a7-8b19-c703f0e190da_ssg_obj_caption_gpt_00000000', 'scan_id': '352e9c30-69fb-27a7-8b19-c703f0e190da', 'target_id': '47', 'instance_type': 'light', 'utterance': 'The light in the image is a bright, soft, and diffused yellow light that is reflected from a window. It has a spherical shape and is made of glass. It can be used to light up a room or to light up a subject, a

=== 3RScan/annotations/scene_cap.json  (13680287 bytes) ===
  dict, 1381 keys, first keys: ['1776ad80-4db7-2333-8b18-f02ef42f3569', 'bf9a3dd3-45a5-2e80-8163-8a5b2573aca4', '4e858c89-fd93-2cb4-8459-7542184fd2ad', 'dc42b378-8d5c-2d2a-8477-c1f077da4e56', '8e0f1c2f-9e28-2339-85ae-05fc50d1a3a7', 'bf9a3dcf-45a5-2e80-8196-c38271d91bcf']
  obj['1776ad80-4db7-2333-8b18-f02ef42f3569'] = {'captions': ['In this scene, there is a tall chair on a wooden floor. The floor is br

## 3. `scannet_scene_cap.json` -- scene count, captions/scene, one example

In [4]:
scannet_cap = json.load(open(SCANNET_SCENE_CAP))

n_scenes = len(scannet_cap)
n_captions = sum(len(v["captions"]) for v in scannet_cap.values())
print(f"scenes: {n_scenes}")
print(f"total captions: {n_captions}")
print(f"avg captions/scene: {n_captions / n_scenes:.2f}")

example_id = next(iter(scannet_cap))
print(f"\nexample scene_id: {example_id}")
print("example caption:", scannet_cap[example_id]["captions"][0][:400])

scenes: 1510
total captions: 4530
avg captions/scene: 3.00

example scene_id: scene0442_00
example caption: In the opulent living room, adorned with four chairs, four tables, and five armchairs, a symphony of elegance unfolds. The chairs, positioned in front of the tables, create an inviting space for conversation and relaxation. The tables, in turn, stand proudly behind the chairs, offering a surface for books, drinks, or cherished mementos. The armchairs, scattered throughout the room, beckon weary so


## 4. Cross-reference `scannet_scene_cap` scene_ids against local posed_images

How many of these scene_ids actually have frames on disk (`source_data/scannet/posed_images/<scene_id>/`)?

In [5]:
cap_ids = set(scannet_cap.keys())
posed_ids = {p.name for p in SCANNET_POSED.iterdir() if p.is_dir()} if SCANNET_POSED.exists() else set()

overlap = cap_ids & posed_ids
print(f"scannet_scene_cap scene_ids: {len(cap_ids)}")
print(f"posed_images scene dirs on disk: {len(posed_ids)}")
print(f"overlap (usable now): {len(overlap)}")
print(f"caption-only, no frames found: {len(cap_ids - posed_ids)}")
if cap_ids - posed_ids:
    print("  example missing:", sorted(cap_ids - posed_ids)[:5])

scannet_scene_cap scene_ids: 1510
posed_images scene dirs on disk: 1513
overlap (usable now): 1510
caption-only, no frames found: 0


## 5. Compare against `leo_annotations`'s 3RScan `scene_caption` (already confirmed earlier)

Reload leo's train+val 3RScan scene captions here too, so this notebook alone
gives the full tally without cross-referencing an older chat message.

In [6]:
leo_3rscan_files = {
    "train": LEO_ANNO / "alignment/scene_caption/3rscan_scenecap_train.json",
    "val": LEO_ANNO / "alignment/scene_caption/3rscan_scenecap_val.json",
}

leo_3rscan = {}
for split, path in leo_3rscan_files.items():
    if not path.exists():
        print(f"[skip] {split}: not found at {path}")
        continue
    data = json.load(open(path))
    leo_3rscan.update(data)
    print(f"{split}: {len(data)} scenes, {sum(len(v) for v in data.values())} caption entries")

print(f"\nleo 3RScan total unique scenes (train+val union): {len(leo_3rscan)}")

# cross-check against raw 3RScan scan dirs on disk
rscan_dirs = {p.name for p in THREERSCAN_ROOT.iterdir() if p.is_dir()} if THREERSCAN_ROOT.exists() else set()
print(f"3RScan scan dirs on disk: {len(rscan_dirs)}")
print(f"leo caption <-> raw scan overlap: {len(set(leo_3rscan) & rscan_dirs)}")

train: 1178 scenes, 17358 caption entries
val: 203 scenes, 3069 caption entries

leo 3RScan total unique scenes (train+val union): 1381
3RScan scan dirs on disk: 1381
leo caption <-> raw scan overlap: 1381


## 5b. SceneVerse's own 3RScan `scene_cap.json` (inside the zip) vs leo's

`3RScan/annotations/scene_cap.json` has 1381 keys -- same count as leo's union.
Check they're the same scenes, how many captions per scene, and whether frames exist on disk.
Run after sections 3 and 5 (uses `leo_3rscan`, `rscan_dirs`).

In [ ]:
with zipfile.ZipFile(THREERSCAN_ZIP) as z:
    sv_3rscan = json.loads(z.read("3RScan/annotations/scene_cap.json"))

sv_ids, leo_ids = set(sv_3rscan), set(leo_3rscan)
sv_caps = sum(len(v["captions"]) for v in sv_3rscan.values())
print(f"sceneverse 3RScan: {len(sv_ids)} scenes, {sv_caps} captions (avg {sv_caps/len(sv_ids):.2f}/scene)")
print(f"overlap with leo scene_caption: {len(sv_ids & leo_ids)}")
print(f"sceneverse-only: {len(sv_ids - leo_ids)}   leo-only: {len(leo_ids - sv_ids)}")
print(f"sceneverse scenes with raw frames on disk: {len(sv_ids & rscan_dirs)}")

## 6. Final tally

Pulls together every scene-level-caption source checked so far (this
notebook + the MMScan `Caption_region` numbers already confirmed via the
remote report) into one summary. Update the `MMSCAN_*` constants below if
those numbers have changed since.

In [ ]:
# MMScan Caption_region numbers from the earlier remote inspection (region-level,
# spans scannet + 3rscan + matterport3d, so mostly overlaps the scenes counted below).
MMSCAN_REGION_TRAIN = 4667
MMSCAN_REGION_VAL = 1191

rscan_union = set(leo_3rscan) | set(sv_3rscan)
scannet_usable = set(scannet_cap) & posed_ids
leo_caps = sum(len(v) for v in leo_3rscan.values())

print("=== Scene-level caption pool (real scans, frames on disk) ===")
print(f"3RScan  unique scenes (leo U sceneverse):  {len(rscan_union & rscan_dirs):>6}")
print(f"ScanNet unique scenes (sceneverse):        {len(scannet_usable):>6}")
print(f"TOTAL unique scenes:                       {len(rscan_union & rscan_dirs) + len(scannet_usable):>6}")
print()
print("caption texts:")
print(f"  leo 3RScan scene_caption:     {leo_caps:>6}")
print(f"  sceneverse 3RScan scene_cap:  {sv_caps:>6}")
print(f"  sceneverse ScanNet scene_cap: {n_captions:>6}")
print(f"  total:                        {leo_caps + sv_caps + n_captions:>6}")
print()
print(f"MMScan Caption_region (region-level, all 3 datasets, train+val): {MMSCAN_REGION_TRAIN + MMSCAN_REGION_VAL} entries")
